In [2]:
'''data extraction'''

import pandas as pd
import numpy as np

f = open('kazuki_data.json')
kazuki_data = pd.read_json(f, orient='split')
f.close()
f = open('watara_data.json')
watara_data = pd.read_json(f, orient='split')
f.close()
f = open('yokoi_data.json')
yokoi_data = pd.read_json(f, orient='split')
f.close()

training = pd.concat([kazuki_data, yokoi_data.loc[0,:].iloc[:2000], yokoi_data.loc[1,:].iloc[:200]])
validation = pd.concat([watara_data, yokoi_data.loc[0,:].iloc[2000:], yokoi_data.loc[1,:].iloc[200:]])

t_healthy = training.loc[0]
t_cancer = training.loc[1]
v_healthy = validation.loc[0]
v_cancer = validation.loc[1]

m_t_healthy = t_healthy.shape[0]
m_t_cancer = t_cancer.shape[0]
m_v_healthy = v_healthy.shape[0]
m_v_cancer = v_cancer.shape[0]

n_mirnas = t_healthy.shape[1] #gets number of columns
miRNA_names_list = list(t_healthy.columns)

print("healthy training data")
print(t_healthy.shape)
print(t_healthy.head)
print("cancer training data")
print(t_cancer.shape)
print(t_cancer.head)
print("healthy validation data")
print(v_healthy.shape)
print(v_healthy.head)
print("cancer validation data")
print(v_cancer.shape)
print(v_cancer.head)

healthy training data
(6965, 2565)
<bound method NDFrame.head of     MIMAT0000062  MIMAT0000063  MIMAT0000064  MIMAT0000065  MIMAT0000066  \
0          3.700         3.669        -2.681         3.499        -2.681   
0         -4.018        -4.018        -4.018        -4.018        -4.018   
0          1.514         0.869         0.464        -0.987        -4.325   
0         -4.417        -4.417        -4.417        -1.036        -1.897   
0         -5.541         0.598        -5.281        -3.185         0.986   
..           ...           ...           ...           ...           ...   
0         -3.515         1.626         6.450        -3.515         1.136   
0         -3.160        -3.160        -3.160        -3.160        -3.160   
0         -3.717         0.509        -3.717        -3.717         1.166   
0         -2.374        -2.374         1.426         2.128         3.553   
0         -5.741         1.007        -5.741        -5.741        -5.741   

    MIMAT0000067  MIMA

In [3]:
t_both = pd.concat((t_healthy, t_cancer))
v_both = pd.concat((v_healthy, v_cancer))

t_index = np.concat((np.zeros(m_t_healthy),np.ones(m_t_cancer)))
Xt_df_unshuffled = pd.DataFrame(t_both, index = t_index)
Xt_df = Xt_df_unshuffled.sample(frac=1, random_state=67)
yt = pd.Series(list(Xt_df.index))
Xt_df = Xt_df.reset_index().drop('index', axis=1)
Xt = Xt_df.values
Xt = Xt.argsort(axis=1)


v_index = np.concat((np.zeros(m_v_healthy),np.ones(m_v_cancer)))
Xv_df_unshuffled = pd.DataFrame(v_both, index = v_index)
Xv_df = Xv_df_unshuffled.sample(frac=1, random_state=67)
yv = pd.Series(list(Xv_df.index))
Xv_df = Xv_df.reset_index().drop('index', axis=1)
Xv = Xv_df.values
Xv = Xv.argsort(axis=1)


In [4]:
Xt, yt, Xv, yv

(array([[   0, 1449, 1448, ..., 1675, 2379, 1400],
        [1282, 1187, 1185, ..., 2480, 2379, 1400],
        [1282, 1862, 1864, ..., 2379, 2480, 1400],
        ...,
        [   0,  863,  865, ..., 2379, 2480, 1400],
        [1282, 1219, 1216, ..., 1329, 1347, 1400],
        [   0, 1167, 1165, ..., 2480, 2379, 1400]], shape=(7165, 2565)),
 0       0.0
 1       0.0
 2       0.0
 3       0.0
 4       0.0
        ... 
 7160    1.0
 7161    0.0
 7162    0.0
 7163    0.0
 7164    0.0
 Length: 7165, dtype: float64,
 array([[1910, 2163, 2131, ..., 1578, 1347, 1400],
        [1282,  993,  997, ..., 2480, 1853, 1400],
        [   0,  969,  977, ..., 2480, 2379, 1400],
        ...,
        [2564, 1949, 1948, ..., 1675, 2379, 1400],
        [1282, 1437, 1436, ..., 1125, 2379, 1400],
        [2564,  652,  653, ..., 1675, 2480, 1400]], shape=(1048, 2565)),
 0       1.0
 1       0.0
 2       0.0
 3       0.0
 4       0.0
        ... 
 1043    1.0
 1044    1.0
 1045    0.0
 1046    0.0
 1047    0.0
 

In [5]:
#logistic regression with lasso
import matplotlib.pyplot as plt

from sklearn.svm import SVC
from sklearn.feature_selection import RFE, RFECV, SelectKBest, f_classif
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import BaggingClassifier
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

import scipy.stats
import math

# not importing scaling bc all of it is binary

OV_PREV = 0.0004

In [5]:
svc = SVC(C=0.01,kernel='linear',class_weight=None)
svc.fit(Xt,yt)

,C,0.01
,kernel,'linear'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [ ]:
selector = SelectKBest(f_classif, k=500)
Xt_select = selector.fit_transform(Xt,yt)

In [60]:
clf_l1_LR = LogisticRegression(C=9e-7, penalty="l1", tol=0.0002, max_iter=5000,solver="saga", class_weight='balanced',verbose=True)
clf_l1_LR.fit(Xt_select,yt)


Epoch 1, change: 1
Epoch 2, change: 1.2197856
Epoch 3, change: 0.22403537
Epoch 4, change: 0.30828123
Epoch 5, change: 0.32848097
Epoch 6, change: 1.8115677
Epoch 7, change: 0.71971538
Epoch 8, change: 0.5099627
Epoch 9, change: 0.84196418
Epoch 10, change: 0.47595945
Epoch 11, change: 0.48017066
Epoch 12, change: 0.71564172
Epoch 13, change: 0.22048762
Epoch 14, change: 0.36373163
Epoch 15, change: 0.7135029
Epoch 16, change: 0.43360307
Epoch 17, change: 0.58625783
Epoch 18, change: 0.89460898
Epoch 19, change: 0.70749117
Epoch 20, change: 0.63048419
Epoch 21, change: 0.40763659
Epoch 22, change: 0.62595146
Epoch 23, change: 0.5994027
Epoch 24, change: 0.65046915
Epoch 25, change: 0.19054718
Epoch 26, change: 1.3254714
Epoch 27, change: 0.60393775
Epoch 28, change: 0.65705439
Epoch 29, change: 1.23522
Epoch 30, change: 0.44851517
Epoch 31, change: 0.72436029
Epoch 32, change: 0.58928321
Epoch 33, change: 0.59041548
Epoch 34, change: 0.92860738
Epoch 35, change: 0.45483877
Epoch 36, ch

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:   42.1s finished


,penalty,'l1'
,dual,False
,tol,0.0002
,C,9e-07
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'saga'
,max_iter,5000
,multi_class,'deprecated'


In [61]:
coef_l1_LR = clf_l1_LR.coef_
sparsity_l1_LR = (coef_l1_LR != 0).sum() 
y_pred = clf_l1_LR.predict(Xt_select)

In [62]:
cm = confusion_matrix(yt,y_pred)
    #sens is tp/(tp+fn)
sensitivity = cm[1,1] / (cm[1,1] + cm[1,0])
#spec is tn/(tn+fp)
specificity = cm[0,0]/ (cm[0,0] + cm[0,1])

print(f"{'Sparsity with L1 penalty:':<40} {sparsity_l1_LR:.2f}%")
print("Confusion Matrix: \n", cm)
print(f"Sensitivity (Recall): {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(classification_report(yt,y_pred))

Sparsity with L1 penalty:                14.00%
Confusion Matrix: 
 [[6919   46]
 [ 182   18]]
Sensitivity (Recall): 0.0900
Specificity: 0.9934
              precision    recall  f1-score   support

         0.0       0.97      0.99      0.98      6965
         1.0       0.28      0.09      0.14       200

    accuracy                           0.97      7165
   macro avg       0.63      0.54      0.56      7165
weighted avg       0.96      0.97      0.96      7165



In [ ]:
rfe = RFE(estimator=svc, n_features_to_select=4, step=10)
model = rfe.fit(new_selection,yt)

In [ ]:
y_pred = model.predict(Xt)
cm = confusion_matrix(y, y_prediction_xvalues>0)
sensitivity = cm[1,1]/(cm[1,1]+cm[1,0])
specificity = (cm[0,0]/(cm[0,0]+cm[0,1]))
print("Confusion Matrix: \n", cm)
print(f"Sensitivity (Recall): {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")

print(classification_report(y,y_prediction_values>0))